In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: env_cleanup (parameterized)
# ⚠️ DANGEROUS: Is env ki saari tables + checkpoints udd jaate hain!
# Stage me chalao to sirf stage schemas, prod me to prod schemas —
# kyunki sab config_setup se aata hai.

from pyspark.sql import SparkSession
import datetime as dt

spark = SparkSession.builder.getOrCreate()

print("💣 Starting Env Cleanup (Dropping all tables & checkpoints of THIS env)...\n")

# ============================================================
# 1. DROP ALL TABLES (env ke 3 schemas)
# ============================================================
schemas_to_clean = [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]

def drop_all_tables(schema_name):
    print(f"🧹 Dropping tables in schema: {schema_name}")
    try:
        tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{schema_name}").collect()
        
        if not tables:
            print(f"   ℹ️ No tables found in {schema_name}")
            return

        for row in tables:
            table_name = row['tableName']
            full_table_name = f"{CATALOG}.{schema_name}.{table_name}"
            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
            print(f"   🗑️ Dropped: {full_table_name}")
            
    except Exception as e:
        print(f"   ❌ Error dropping tables in {schema_name}: {e}")

for schema in schemas_to_clean:
    drop_all_tables(schema)
    print("-" * 50)

# ============================================================
# 2. DELETE ALL CHECKPOINTS (sirf is env ke)
# ============================================================
checkpoint_base_path = CHECKPOINTS

print(f"\n🧹 Deleting all checkpoints at: {checkpoint_base_path}")
try:
    dbutils.fs.ls(checkpoint_base_path)
    dbutils.fs.rm(checkpoint_base_path, recurse=True)
    print(f"   🗑️ Successfully deleted ALL checkpoints!")
except Exception as e:
    print(f"   ℹ️ Nothing to delete or folder doesn't exist: {e}")

print(f"\n✅ ENV CLEANUP COMPLETE! Fresh start kar sakte ho.")